In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
src_dir = cwd / "src" if (cwd / "src").exists() else cwd.parent / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from utils.io import (
    list_containers,
    explore_lif_container,
    load_lif_image,
    calculate_rescale_factor,
    get_voxel_spacing_zyx_um,
    ensure_output_dir,
    load_precomputed_results_if_available,
)
from utils.segmentation import (
    predict_nuclei_labels,
    simulate_fluo_from_bf,
    generate_rough_root_3d,
    fill_root_3d,
    smooth_outer_root_surface_3d,
    wrap_outer_root_surface
)
from utils.inference import predict_tiled_unet

from utils.feature_extraction import (
    calculate_distance_to_root_surface,
    extract_nuclei_features_per_marker,
    compute_fret_ratios,
    classify_root_cap_nuclei,
    extract_nuclei_depth,
    map_root_body_depth_clusters_to_tissue_layers,
    merge_root_cap_into_tissue_layers
)

from utils.data_viz import map_df_column_to_labels

import tifffile
import napari
import torch

In [ ]:
# Copy the path where your .lif containers are stored, you can use absolute or relative paths to point at other disk locations
RAW_DATA_DIRECTORY = r"../raw_data"

# Point to the local PanSeg model UNet3D weights and config files
# You can choose from lightsheet_3D_unet_root_ds1x, lightsheet_3D_unet_root_ds2x, lightsheet_3D_unet_root_ds3x, confocal_3D_unet_sa_meristem_cells & generic_confocal_3D_unet
MODEL_DIR = "../plantseg_models/lightsheet_3D_unet_root_ds3x"

# Channel index used for CellposeSAM-based 3D nuclei segmentation
NUCLEI_CHANNEL = 2

# Minimum and maximum nuclei label volume to use for filtering predicted nuclei labels
MIN_MAX_NUCLEI_VOLUME = (250, 4000)

# FRET-based biosensor system (nlsABACUS2-400)
# edCerulean_CTRL: excitation of edCerulean, emission of edCerulean (donor, DD)
# edCitrine_FRET: excitation of edCerulean (donor), emission of edCitrine (acceptor, DA – FRET signal)
# edCitrine_CTRL: excitation of edCitrine, emission of edCitrine (acceptor, AA) – used for nuclei segmentation (and optionally for correction)
MARKERS = (("edCerulean_CTRL", 0, "DD"), ("edCitrine_FRET", 1, "DA"), ("edCitrine_CTRL", 2, "AA"), ("brightfield", 3, "root_structure"))

# Mark the position of the .lif container you want to open in your raw data folder (first = 0, second = 1, third = 2)
LIF_CONTAINER_INDEX = 0 

# Mark the position of the image inside the .lif container you want to open (first = 0, second = 1, third = 2)
LIF_IMAGE_INDEX = 2

In [ ]:
# Iterate through the .lif container files in the directory
lif_containers = list_containers(RAW_DATA_DIRECTORY, file_format="lif")

if not lif_containers:
    raise FileNotFoundError(
        f"No .lif containers found in '{RAW_DATA_DIRECTORY}'. "
        "Check RAW_DATA_DIRECTORY and file extension."
    )

if not 0 <= LIF_CONTAINER_INDEX < len(lif_containers):
    raise IndexError(
        f"LIF_CONTAINER_INDEX={LIF_CONTAINER_INDEX} is out of range. "
        f"Valid range is 0..{len(lif_containers)-1}."
    )

lif_containers

In [ ]:
# Explore different .lif files (0 defines the first file in the directory)
lif_path = lif_containers[LIF_CONTAINER_INDEX]

# Explore the contents of a single .lif container
nr_imgs, lif_container_id = explore_lif_container(file_path=lif_path, display=True)

# Load a single image from a .lif container
lif_image, lif_image_name, xml_metadata = load_lif_image(file_path=lif_path, image_index=LIF_IMAGE_INDEX)

In [ ]:
# Initialize Napari Viewer and display all input channels
viewer = napari.Viewer(ndisplay=3)
viewer.add_image(lif_image, 
                channel_axis=0,
                colormap=['cyan', 'yellow', 'magenta', 'inferno'],
                name=[tuple[0] for tuple in MARKERS] #['edCerulean_CTRL','edCitrine_FRET','edCitrine_CTRL','brightfield']
                )

In [ ]:
# Simulate fluorescently labelled cell walls from brightfield input image
sim_fluo_cell_walls = simulate_fluo_from_bf(lif_image, MARKERS)

# Add simulated fluorescently labelled plant cell boundaries to Napari viewer
viewer.add_image(sim_fluo_cell_walls,
                name="PanSeg_UNet3D_input",
                colormap="gray",
                blending="additive")


In [ ]:
# Ensure output directory for this container's nuclei labels
nuclei_labels_dir = ensure_output_dir(RAW_DATA_DIRECTORY, lif_container_id, results_type="nuclei_labels")
print(f"Nuclei labels directory: {nuclei_labels_dir}")

# Calculate anisotropy CellposeSAM parameter to rescale across the Z-axis (ratio of Z-resolution to XY-resolution)
rescale_factor = calculate_rescale_factor(xml_metadata, display=True)

# Load precomputed labels when available; otherwise predict and store them
nuclei_labels = load_precomputed_results_if_available(nuclei_labels_dir, lif_image_name, results_type="nuclei_labels")

if nuclei_labels is not None:
    print(f"Predictions already calculated for: {lif_image_name} ...loading")
    # Add the prediction to the napari viewer
    viewer.add_labels(nuclei_labels)
    
else:
    # Predict nuclei labels using CellposeSAM using anisotropy correction
    nuclei_labels = predict_nuclei_labels(lif_image, rescale_factor, NUCLEI_CHANNEL, MIN_MAX_NUCLEI_VOLUME, visualize=True)
    # Create path for nuclei labels (used only when saving a newly computed prediction)
    nuclei_labels_path = nuclei_labels_dir / f"{lif_image_name}_nuclei_labels.tif"
    # Save the prediction
    tifffile.imwrite(nuclei_labels_path, nuclei_labels)



In [ ]:
# Create a dictionary containing all image descriptors
descriptor_dict = {
            "lif_container_id": lif_container_id,
            "lif_image_name": lif_image_name,
            }

# Extract morphological and intensity features per marker
props_df = extract_nuclei_features_per_marker(nuclei_labels, lif_image, MARKERS, descriptor_dict)

# Calculate FRET ratios and add to props_df
props_df = compute_fret_ratios(props_df, MARKERS)

In [ ]:
# Ensure output directory for this container's 3D root mask
root_mask_dir = ensure_output_dir(RAW_DATA_DIRECTORY, lif_container_id, results_type="root_mask")
print(f"3D Root Mask directory: {root_mask_dir}")

# Load precomputed root mask when available; otherwise generate and store it
root_body_3d = load_precomputed_results_if_available(root_mask_dir, lif_image_name, results_type="root_mask")

if root_body_3d is not None:
    print(f"3D root mask already calculated for: {lif_image_name} ...loading")
    # Classify nuclei as belonging to the root cap or the rest of the root structure using clustering.
    props_df = classify_root_cap_nuclei(props_df)
    # Add the precomputed mask to the napari viewer
    viewer.add_image(root_body_3d, name="smooth_root_3d", colormap="green", blending="additive", opacity=0.5)

else:
    # Predict root cell boundary probability maps using a pre-trained UNet3D model
    root_pmaps = predict_tiled_unet(
        raw=sim_fluo_cell_walls,
        input_layout="ZYX",
        model_dir=MODEL_DIR,
        patch=(80, 160, 160),
        patch_halo=(8, 16, 16),
        stride_ratio=0.75,
        batch_size=1,
        device="cuda" if torch.cuda.is_available() else "cpu",
        use_amp=True,
    )

    # root_pmaps: (C_out, Z, Y, X)
    viewer.add_image(root_pmaps[0], name="root_unet_pmap", colormap="viridis", blending="additive")

    # Generate a rough 3D root mask
    filled_3d_closed = generate_rough_root_3d(root_pmaps, nuclei_labels, probability_threshold=0.9, visualize=True)
    # Fill internal gaps inside rough 3D root mask
    filled_root_3d = fill_root_3d(filled_3d_closed, occupancy_threshold=0.9, slice_aware_filling=True, visualize=True)
    # Smooth root outer surface to remove small protrusions
    smooth_root_3d = smooth_outer_root_surface_3d(filled_root_3d, erosion=5, smoothing=3, visualize=True)
    # Classify nuclei as belonging to the root cap or the rest of the root structure using clustering.
    props_df = classify_root_cap_nuclei(props_df)
    # Wrap the outer root surface so it fits better around the outtermost top nuclei layer (ignore root cap nuclei)
    root_body_3d = wrap_outer_root_surface(nuclei_labels, smooth_root_3d, props_df, percentage_threshold = 5.0, edt_threshold = 15.0, visualize = True)
    # Create path for root mask (used only when saving a newly computed prediction)
    root_mask_path = root_mask_dir / f"{lif_image_name}_root_mask.tif"
    # Save the processed mask
    tifffile.imwrite(root_mask_path, root_body_3d)

In [ ]:
# Ensure output directory for this container's nuclei depth_map
depth_map_dir = ensure_output_dir(RAW_DATA_DIRECTORY, lif_container_id, results_type="depth_map")
print(f"Nuclei depth map directory: {depth_map_dir}")

# Load precomputed depth map when available; otherwise generate and store it
nuclei_depth_map = load_precomputed_results_if_available(depth_map_dir, lif_image_name, results_type="depth_map")
is_flooded = False
flooded_planes = []

if nuclei_depth_map is not None:
    print(f"Nuclei depth map already calculated for: {lif_image_name} ...loading")
    # Add the precomputed map to Napari
    viewer.add_image(nuclei_depth_map, name="nuclei_depth_normalized", colormap="viridis", blending="additive")

else:
    # Always compute depth distances with anisotropy correction using metadata voxel spacing (z, y, x) in um.
    # This affects distance computation for depth assignment, not visualization.
    spacing_zyx_um = get_voxel_spacing_zyx_um(xml_metadata)
    print(f"Using spacing-aware depth distance mode with spacing (z,y,x) um = {spacing_zyx_um}.")

    # Calculate distance from each nuclei centroid to the root surface.
    # This will be used to approximate to which tissue layer each nucleus belongs.
    nuclei_depth_map, is_flooded, flooded_planes = calculate_distance_to_root_surface(
        nuclei_labels,
        root_body_3d,
        pad_full_root=False,
        spacing_zyx_um=spacing_zyx_um,
        visualize=True,
    )

    # Create path for nuclei depth map (used only when saving a newly computed prediction)
    nuclei_depth_path = depth_map_dir / f"{lif_image_name}_depth_map.tif"
    # Save the calculated depth map
    tifffile.imwrite(nuclei_depth_path, nuclei_depth_map)

print(f"Flood fill applied: {is_flooded}; flooded planes: {flooded_planes}")

In [ ]:
# Extract depth information from nuclei labels and merge with props_df
depth_df = extract_nuclei_depth(nuclei_labels, nuclei_depth_map)
props_df = props_df.merge(depth_df, on="label")

# Map root body depth clusters to tissue layers
depth_clusters_df = map_root_body_depth_clusters_to_tissue_layers(props_df)

# Merge root cap into tissue layers
props_df = merge_root_cap_into_tissue_layers(props_df, depth_clusters_df)

props_df

In [ ]:
# Visualize root cap nuclei
tip_cluster_id_img = map_df_column_to_labels(
    nuclei_labels,
    props_df,
    value_column="tip_cluster_id",
    colormap="turbo",
    visualize=True
)

In [ ]:
# Visualize depth clusters
depth_cluster_id_img = map_df_column_to_labels(
    nuclei_labels,
    props_df,
    value_column="depth_cluster_id",
    colormap="turbo",
    visualize=True
)

In [ ]:
# Visualize FRET ratios
fret_img = map_df_column_to_labels(
    nuclei_labels,
    props_df,
    value_column="FRET_ratio_sum_norm_per_image",
    colormap="inferno",
    visualize=True
)

In [ ]:
#TODO: Allow researcher to convert depth_cluster_id_img to labels for editing, and save the results as .tif. Include the same logic as for nuclei_labels, depth_map (but this time for depth_cluster_id)
# if results are pre-computed these are loaded from disk. In this way the manual changes applied by the researcher can influence props_df (need to write this logic, not implemented atm). The logic should overwrite the depth_cluster_id field in props_df according to the label identifier contained in depth_cluster_id_img.